# Abell 3411 jax-lensing posterior — exact SNR with 3200 chains

Uses the full `kappa_E_samples` array now saved by `abell3411_jaxlensing.py` to:
* Verify MC convergence directly (no need to guess from mean/std alone).
* Compute the **exact** 1.5-arcmin-smoothed posterior std by smoothing each of the 3200 chains, then taking the std across chains — replacing the optimistic/conservative bracket we had previously.
* Reproject everything onto the KS WCS for direct side-by-side comparison with your colleague's KS SNR map.
* Save the resulting maps as FITS for DS9 overlays.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.ndimage import gaussian_filter

from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp

POSTERIOR_NPZ = '/projects/mccleary_group/habjan.e/TNG/Data/jaxlense_dataset/posterior_abell3411.npz'
KS_FITS       = '/home/habjan.e/TNG/Data/superbit_redshifts/Abell_3411/snr_ks_Abell3411_b.fits'
OUT_DIR       = '/home/habjan.e/TNG/Data/superbit_redshifts/Abell_3411'

d = np.load(POSTERIOR_NPZ)

kE_mean = d['kappa_E_mean']
kE_std  = d['kappa_E_std']
samples = d['kappa_E_samples']                 # (N_chains, 128, 128)
g1_obs  = d['gamma1_obs']
g2_obs  = d['gamma2_obs']
n_eff   = d['n_eff_pix']
mask    = d['mask']

N_CHAINS, MAP_SIZE, _ = samples.shape

half_fov_arcmin = float(d['half_fov_arcmin'])
pix_arcmin_src  = float(d['pixel_size_arcmin_data'])
z_l             = float(d['z_l'])
ra0, dec0       = float(d['ra0']), float(d['dec0'])
fov_mpc         = float(d['fov_mpc'])
mb_a3411        = float(d['mean_beta_a3411'])
mb_train        = float(d['mean_beta_training'])
sigma_e_eff     = float(d['sigma_e_eff_per_component'])
Sigma_crit_inf  = float(d['sigma_crit_inf_a3411'])
n_post          = int(d['n_posterior_samples'])

EXTENT_SRC = (-half_fov_arcmin, half_fov_arcmin, -half_fov_arcmin, half_fov_arcmin)

print(f'cluster centre   : (RA, Dec) = ({ra0:.4f}, {dec0:.4f}) deg')
print(f'z_l              : {z_l}')
print(f'source grid      : {MAP_SIZE} x {MAP_SIZE}, pix = {pix_arcmin_src:.4f} arcmin')
print(f'posterior chains : {N_CHAINS} (samples array size = {samples.nbytes / 1e6:.1f} MB)')
print(f'kE_mean range    : [{kE_mean.min():+.4f}, {kE_mean.max():+.4f}]')
print(f'kE_std  range    : [{kE_std.min():.4f}, {kE_std.max():.4f}]')
print(f'data coverage    : {int(mask.sum())}/{mask.size} pixels ({100*mask.sum()/mask.size:.1f}%)')

## 1. MC convergence diagnostic

Two checks using the full sample stack:
* **Split-half test**: compute per-pixel mean from chains 0..N/2 and N/2..N independently. If chains are i.i.d. samples from the posterior, the difference at each pixel should be a draw from N(0, σ_κ × √(4/N)). The ratio |Δmean| / σ_MC has expected median 0.674 and 95th percentile 1.96 for the half-normal.
* **Per-pixel histograms** at the cluster centre and an off-cluster pixel. Unimodal = posterior is well-behaved and mean+std are sufficient summaries. Multimodal = mean misleads.

In [ ]:
half1 = samples[:N_CHAINS // 2].mean(axis=0)
half2 = samples[N_CHAINS // 2:].mean(axis=0)
sigma_MC_diff = kE_std * np.sqrt(4.0 / N_CHAINS)
ratio = np.abs(half1 - half2) / np.maximum(sigma_MC_diff, 1e-10)

print('Split-half convergence (ratio = |half1 - half2| / sigma_MC_diff):')
print(f'  median         = {np.median(ratio):.3f}     (expected 0.674 for converged i.i.d.)')
print(f'  95th percentile = {np.percentile(ratio, 95):.3f}     (expected 1.96)')
print(f'  max            = {np.max(ratio):.3f}')
if np.median(ratio) < 1.0 and np.percentile(ratio, 95) < 3.0:
    print('  -> CHAINS ARE WELL-MIXED. More samples will not reduce per-pixel mean noise.')
else:
    print('  -> CHAINS NOT CONVERGED. Increase n_seeds or num_steps_between_results.')

# Per-pixel histograms at centre + off-cluster ref pixel
centre_ix = MAP_SIZE // 2; centre_iy = MAP_SIZE // 2
off_ix = MAP_SIZE // 2 + 20; off_iy = MAP_SIZE // 2 + 20

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, ix, iy, label in [(axes[0], centre_ix, centre_iy, 'cluster centre'),
                          (axes[1], off_ix,    off_iy,    f'off-cluster (dRA, dDec = +{(off_ix-MAP_SIZE//2)*pix_arcmin_src:.1f}, +{(off_iy-MAP_SIZE//2)*pix_arcmin_src:.1f} arcmin)')]:
    vals = samples[:, iy, ix]
    ax.hist(vals, bins=40, density=True, alpha=0.7)
    ax.axvline(vals.mean(), color='r', label=f'mean = {vals.mean():+.4f}')
    ax.axvline(vals.mean() + vals.std(), color='r', ls='--', alpha=0.6, label=f'±1σ = {vals.std():.4f}')
    ax.axvline(vals.mean() - vals.std(), color='r', ls='--', alpha=0.6)
    ax.set_xlabel(r'$\kappa_E$'); ax.set_ylabel('p.d.f.')
    ax.set_title(f'Posterior at pixel ({ix}, {iy})\n{label}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Exact smoothed posterior moments via per-chain smoothing

Smooth each of the 3200 chain images with a 1.5-arcmin FWHM Gaussian (the same kernel as KS), then compute the mean and std across smoothed chains. The std is now exact — it captures the cross-pixel correlations that the prior enforces, with no independence assumption.

In [ ]:
FWHM_arcmin  = 1.5
sigma_arcmin = FWHM_arcmin / 2.3548
sigma_pix_src = sigma_arcmin / pix_arcmin_src
print(f'kernel: FWHM = {FWHM_arcmin} arcmin -> sigma = {sigma_arcmin:.3f} arcmin = {sigma_pix_src:.3f} src pixels')

import time
t0 = time.time()
# Smooth each chain (in source frame, 128x128). gaussian_filter is fast; this is ~5s for 3200 maps.
samples_smooth = np.empty_like(samples)
for k in range(N_CHAINS):
    samples_smooth[k] = gaussian_filter(samples[k], sigma=sigma_pix_src)
print(f'  smoothed {N_CHAINS} chains in {time.time() - t0:.1f} s')

kE_s_src    = samples_smooth.mean(axis=0).astype(np.float32)
kEstd_s_src = samples_smooth.std(axis=0).astype(np.float32)
snr_s_src   = kE_s_src / np.where(kEstd_s_src > 0, kEstd_s_src, 1.0)

print(f'smoothed kE_mean range: [{kE_s_src.min():+.4f}, {kE_s_src.max():+.4f}]')
print(f'smoothed kE_std  range: [{kEstd_s_src[kEstd_s_src>0].min():.4f}, {kEstd_s_src.max():.4f}]')
print(f'reduction factor in std from smoothing: median = {np.median(kE_std) / np.median(kEstd_s_src):.2f}x')
print(f'                                          (independence limit would be sqrt(4 pi sigma_pix^2) = {np.sqrt(4*np.pi*sigma_pix_src**2):.2f}x)')
print(f'exact smoothed SNR range: [{snr_s_src[mask>0].min():+.2f}, {snr_s_src[mask>0].max():+.2f}]')

The std reduction is the diagnostic: if it's close to √(4π σ²) (the independent-pixel limit), the posterior pixels are nearly uncorrelated and the prior is doing little smoothing. If it's close to 1 (no reduction), the posterior is fully correlated and the prior is acting like a strong smoother. The truth between these tells us how much information the prior is actually adding.

## 3. WCS setup and reprojection onto the KS frame

Build a source WCS for the jax-lensing 128×128 grid, load the target WCS from `snr_ks_Abell3411_b.fits`, and reproject all the maps we care about onto the KS pixel grid.

In [ ]:
pix_deg_src = pix_arcmin_src / 60.0

src_wcs = WCS(naxis=2)
src_wcs.wcs.crval = [ra0, dec0]
src_wcs.wcs.crpix = [(MAP_SIZE + 1) / 2.0, (MAP_SIZE + 1) / 2.0]
src_wcs.wcs.cdelt = [+pix_deg_src, +pix_deg_src]
src_wcs.wcs.ctype = ['RA---TAN', 'DEC--TAN']
src_wcs.wcs.radesys = 'ICRS'

with fits.open(KS_FITS) as hdul:
    ks_data = hdul[0].data.astype(np.float64)
    tgt_wcs = WCS(hdul[0].header)
    tgt_shape = hdul[0].data.shape

print(f'src WCS shape={MAP_SIZE}^2, pix={abs(src_wcs.wcs.cdelt[0])*3600:.2f}"')
print(f'tgt WCS shape={tgt_shape}, pix={abs(tgt_wcs.wcs.cdelt[0])*3600:.2f}"')

def reproj(arr, order='bilinear'):
    out, _ = reproject_interp((arr.astype(np.float64), src_wcs), tgt_wcs,
                              shape_out=tgt_shape, order=order)
    return out

# Unsmoothed quantities
kE_r    = reproj(kE_mean)
kEstd_r = np.sqrt(np.maximum(reproj(kE_std ** 2), 0))   # reproject variance, then sqrt
mask_r  = reproj(mask, order='nearest-neighbor')
neff_r  = reproj(n_eff)

# Exact-smoothed quantities (variance reprojected for std)
kE_s_r    = reproj(kE_s_src)
kEstd_s_r = np.sqrt(np.maximum(reproj(kEstd_s_src ** 2), 0))

for arr in (kE_r, kEstd_r, mask_r, neff_r, kE_s_r, kEstd_s_r):
    np.nan_to_num(arr, copy=False, nan=0.0)
mask_r = (mask_r > 0.5).astype(np.float32)

snr_r   = kE_r   / np.where(kEstd_r   > 0, kEstd_r,   1.0)
snr_s_r = kE_s_r / np.where(kEstd_s_r > 0, kEstd_s_r, 1.0)

print(f'reprojected smoothed kE   range: [{kE_s_r.min():+.4f}, {kE_s_r.max():+.4f}]')
print(f'reprojected smoothed std  range: [{kEstd_s_r[kEstd_s_r>0].min():.4f}, {kEstd_s_r.max():.4f}]')
print(f'reprojected smoothed SNR  range: [{snr_s_r[mask_r>0].min():+.2f}, {snr_s_r[mask_r>0].max():+.2f}]')

## 4. Detection summary (single number, exact SNR)

Now that the smoothed σ is the exact posterior std of the smoothed field, the SNR is a single value, not a bracket.

In [ ]:
snr_view = snr_s_r.copy()
snr_view[mask_r < 0.5] = -np.inf
iy_peak, ix_peak = np.unravel_index(np.argmax(snr_view), snr_view.shape)
peak_snr = float(snr_s_r[iy_peak, ix_peak])
peak_radec = tgt_wcs.pixel_to_world_values(ix_peak, iy_peak)
ra_peak, dec_peak = float(peak_radec[0]), float(peak_radec[1])
offset_arcmin = np.hypot((ra_peak - ra0) * np.cos(np.deg2rad(dec0)),
                         dec_peak - dec0) * 60.0

ks_peak_idx = np.unravel_index(np.argmax(ks_data), ks_data.shape)
ks_peak_radec = tgt_wcs.pixel_to_world_values(ks_peak_idx[1], ks_peak_idx[0])
ks_peak_offset = np.hypot((float(ks_peak_radec[0]) - ra0) * np.cos(np.deg2rad(dec0)),
                          float(ks_peak_radec[1]) - dec0) * 60.0

print('=== jax-lensing peak (1.5 arcmin smoothed, inside data footprint) ===')
print(f'  pixel       = ({ix_peak}, {iy_peak})')
print(f'  (RA, Dec)   = ({ra_peak:.4f}, {dec_peak:.4f}) deg')
print(f'  offset from catalog centre = {offset_arcmin:.2f} arcmin')
print(f'  exact SNR   = {peak_snr:+.2f}')
if peak_snr >= 3.0:
    verdict = f'DETECTION at {peak_snr:.1f}sigma'
elif peak_snr >= 2.0:
    verdict = 'marginal: 2-3 sigma'
else:
    verdict = 'no detection at smoothed scale'
print(f'  verdict     : {verdict}')
print()
print('=== KS map peak (reference) ===')
print(f'  (RA, Dec) = ({float(ks_peak_radec[0]):.4f}, {float(ks_peak_radec[1]):.4f}) deg')
print(f'  offset from catalog centre = {ks_peak_offset:.2f} arcmin')
print(f'  KS SNR  = {float(ks_data[ks_peak_idx]):+.2f}')
print()
# SNR at the *KS peak location*, not our own peak — most direct comparison
snr_at_ks_peak = float(snr_s_r[ks_peak_idx])
print(f'jax-lensing SNR evaluated at the KS peak pixel = {snr_at_ks_peak:+.2f}')

## 5. Unsmoothed maps on the KS WCS

In [ ]:
kE_lim  = float(np.percentile(np.abs(kE_r), 99))
snr_lim = float(np.percentile(np.abs(snr_r), 99))

fig = plt.figure(figsize=(17, 5.2), constrained_layout=True)
ax0 = fig.add_subplot(1, 3, 1, projection=tgt_wcs)
ax1 = fig.add_subplot(1, 3, 2, projection=tgt_wcs)
ax2 = fig.add_subplot(1, 3, 3, projection=tgt_wcs)

im0 = ax0.imshow(kE_r, origin='lower', cmap='RdBu_r',
                 norm=TwoSlopeNorm(vcenter=0, vmin=-kE_lim, vmax=+kE_lim))
ax0.set_title(r'$\kappa_E$ posterior mean (unsmoothed)')
plt.colorbar(im0, ax=ax0, fraction=0.046, label=r'$\kappa_E$')

im1 = ax1.imshow(kEstd_r, origin='lower', cmap='magma')
ax1.set_title(r'$\sigma_{\kappa_E}$ (unsmoothed)')
plt.colorbar(im1, ax=ax1, fraction=0.046, label=r'$\sigma_{\kappa_E}$')

im2 = ax2.imshow(snr_r, origin='lower', cmap='RdBu_r',
                 norm=TwoSlopeNorm(vcenter=0, vmin=-snr_lim, vmax=+snr_lim))
ax2.set_title('SNR (unsmoothed, per pixel)')
plt.colorbar(im2, ax=ax2, fraction=0.046, label='SNR')

for ax in (ax0, ax1, ax2):
    ax.contour(mask_r, levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--',
               transform=ax.get_transform('pixel'))
    ax.scatter([ra0], [dec0], transform=ax.get_transform('world'),
               marker='+', c='white', s=120, lw=1.5)
    ax.set_xlabel('RA (J2000)'); ax.set_ylabel('Dec (J2000)')

plt.show()

## 6. Exact-smoothed maps on the KS WCS (1.5 arcmin FWHM)

The σ map here was computed by smoothing each of the 3200 posterior chains and then taking the std across smoothed chains, so it includes the cross-pixel correlations the prior enforces.

In [ ]:
kE_s_lim  = float(np.percentile(np.abs(kE_s_r), 99.5))
snr_s_lim = float(max(np.percentile(np.abs(snr_s_r), 99), 2.0))

fig = plt.figure(figsize=(17, 5.2), constrained_layout=True)
ax0 = fig.add_subplot(1, 3, 1, projection=tgt_wcs)
ax1 = fig.add_subplot(1, 3, 2, projection=tgt_wcs)
ax2 = fig.add_subplot(1, 3, 3, projection=tgt_wcs)

im0 = ax0.imshow(kE_s_r, origin='lower', cmap='RdBu_r',
                 norm=TwoSlopeNorm(vcenter=0, vmin=-kE_s_lim, vmax=+kE_s_lim))
ax0.set_title(r'$\kappa_E$ smoothed (FWHM 1.5 arcmin)')
plt.colorbar(im0, ax=ax0, fraction=0.046, label=r'$\kappa_E$')

im1 = ax1.imshow(kEstd_s_r, origin='lower', cmap='magma')
ax1.set_title(r'$\sigma_{\kappa_E}$ smoothed (exact)')
plt.colorbar(im1, ax=ax1, fraction=0.046, label=r'$\sigma_{\kappa_E}$')

im2 = ax2.imshow(snr_s_r, origin='lower', cmap='RdBu_r',
                 norm=TwoSlopeNorm(vcenter=0, vmin=-snr_s_lim, vmax=+snr_s_lim))
ax2.set_title('SNR smoothed (exact)')
plt.colorbar(im2, ax=ax2, fraction=0.046, label='SNR')

for ax in (ax0, ax1, ax2):
    ax.contour(mask_r, levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--',
               transform=ax.get_transform('pixel'))
    ax.scatter([ra0], [dec0], transform=ax.get_transform('world'),
               marker='+', c='white', s=120, lw=1.5)
    ax.set_xlabel('RA (J2000)'); ax.set_ylabel('Dec (J2000)')

plt.show()

## 7. Side-by-side with KS SNR (the comparison plot)

Both panels on the same WCS, same colour scale. The KS map is pre-smoothed with a 1.5-arcmin kernel and noise-calibrated by your colleague; the jax-lensing map is our exact 1.5-arcmin smoothed SNR.

In [ ]:
ks_lim     = float(np.percentile(np.abs(ks_data), 99))
common_lim = max(ks_lim, snr_s_lim, 3.0)

fig = plt.figure(figsize=(13, 5.5), constrained_layout=True)
axL = fig.add_subplot(1, 2, 1, projection=tgt_wcs)
axR = fig.add_subplot(1, 2, 2, projection=tgt_wcs)

imL = axL.imshow(ks_data, origin='lower', cmap='RdBu_r',
                 norm=TwoSlopeNorm(vcenter=0, vmin=-common_lim, vmax=+common_lim))
axL.set_title('KS SNR (Sayan, kernel = 1.5 arcmin)')
plt.colorbar(imL, ax=axL, fraction=0.046, label='SNR')

imR = axR.imshow(snr_s_r, origin='lower', cmap='RdBu_r',
                 norm=TwoSlopeNorm(vcenter=0, vmin=-common_lim, vmax=+common_lim))
axR.set_title(f'jax-lensing SNR (1.5-arcmin smoothed, exact, {N_CHAINS} chains)')
plt.colorbar(imR, ax=axR, fraction=0.046, label='SNR')

for ax in (axL, axR):
    ax.contour(mask_r, levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--',
               transform=ax.get_transform('pixel'))
    ax.scatter([ra0], [dec0], transform=ax.get_transform('world'),
               marker='+', c='white', s=120, lw=1.5)
    ax.set_xlabel('RA (J2000)'); ax.set_ylabel('Dec (J2000)')

plt.show()

## 8. Export FITS files (KS WCS in header)

Replaces the previous optimistic/conservative SNR FITS files with a single exact-smoothed SNR file. All carry the colleague's WCS so they align in DS9.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
tgt_header = tgt_wcs.to_header()

def save_fits(arr, name, extra_hdr=None):
    h = tgt_header.copy()
    if extra_hdr:
        for k, v in extra_hdr.items():
            h[k] = v
    out_path = os.path.join(OUT_DIR, name)
    fits.writeto(out_path, arr.astype(np.float32), header=h, overwrite=True)
    print(f'  wrote {out_path}  shape={arr.shape}, '
          f'range=[{np.nanmin(arr):+.4g}, {np.nanmax(arr):+.4g}]')

# Clean up the previous optimistic/conservative SNR files if they exist;
# they're superseded by the exact computation.
for legacy in ['a3411_snr_jaxlense_smooth15_lo.fits',
               'a3411_snr_jaxlense_smooth15.fits']:
    p = os.path.join(OUT_DIR, legacy)
    if os.path.exists(p):
        os.remove(p); print(f'  removed legacy {p}')

save_fits(kE_r,      'a3411_kappa_jaxlense.fits',            {'BUNIT':'kappa', 'OBJECT':'Abell3411', 'METHOD':'jaxlensing-posterior-mean', 'NCHAINS':N_CHAINS})
save_fits(kEstd_r,   'a3411_sigma_jaxlense.fits',            {'BUNIT':'sigma_kappa', 'OBJECT':'Abell3411', 'METHOD':'jaxlensing-posterior-std', 'NCHAINS':N_CHAINS})
save_fits(snr_r,     'a3411_snr_jaxlense.fits',              {'BUNIT':'snr', 'OBJECT':'Abell3411', 'METHOD':'per-pixel-snr', 'NCHAINS':N_CHAINS})
save_fits(kE_s_r,    'a3411_kappa_jaxlense_smooth15.fits',   {'BUNIT':'kappa', 'OBJECT':'Abell3411', 'KERNEL':'1.5 arcmin FWHM', 'NCHAINS':N_CHAINS})
save_fits(kEstd_s_r, 'a3411_sigma_jaxlense_smooth15.fits',   {'BUNIT':'sigma_kappa', 'OBJECT':'Abell3411', 'KERNEL':'1.5 arcmin FWHM', 'METHOD':'exact-per-chain-smoothing'})
save_fits(snr_s_r,   'a3411_snr_jaxlense_smooth15.fits',     {'BUNIT':'snr', 'OBJECT':'Abell3411', 'KERNEL':'1.5 arcmin FWHM', 'METHOD':'exact-per-chain-smoothing', 'NCHAINS':N_CHAINS})
save_fits(mask_r,    'a3411_mask_jaxlense.fits',             {'BUNIT':'0/1', 'OBJECT':'Abell3411', 'METHOD':'data-footprint'})

## 9. Source-frame sanity view (arcmin from cluster centre)

Quick check on the original 128×128 grid. East-on-right here vs. east-on-left in the WCS plots above — that flip is expected and handled by the reprojection.

In [ ]:
ZOOM = 15.0
snr_src   = kE_mean / np.where(kE_std > 0, kE_std, 1.0)
snr_s_src_view = snr_s_src.copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), constrained_layout=True)
panels = [
    (axes[0], kE_s_src,    r'$\kappa_E$ smoothed (src frame)', 'RdBu_r',
         TwoSlopeNorm(vcenter=0, vmin=-kE_s_lim, vmax=+kE_s_lim), r'$\kappa_E$'),
    (axes[1], kEstd_s_src, r'$\sigma_{\kappa_E}$ smoothed (exact)', 'magma', None, r'$\sigma_{\kappa_E}$'),
    (axes[2], snr_s_src_view, 'SNR smoothed (exact)', 'RdBu_r',
         TwoSlopeNorm(vcenter=0, vmin=-snr_s_lim, vmax=+snr_s_lim), 'SNR'),
]
for ax, arr, title, cmap, norm, cb_label in panels:
    im = ax.imshow(arr, origin='lower', extent=EXTENT_SRC, cmap=cmap, norm=norm)
    ax.contour(mask, levels=[0.5], extent=EXTENT_SRC, colors='cyan', linewidths=1.0, linestyles='--')
    ax.plot(0, 0, '+', color='white', markersize=12, markeredgewidth=1.5)
    ax.set_xlim(-ZOOM, ZOOM); ax.set_ylim(-ZOOM, ZOOM)
    ax.set_xlabel('dRA [arcmin]'); ax.set_ylabel('dDec [arcmin]')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, label=cb_label)

plt.show()